In [1]:
import os
import numpy as np
import zstandard as zstd
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm import tqdm

# -------------------------------------------------
# Assumes you have an lts object available, e.g.:
# from your_module import lts
# -------------------------------------------------

# -------------------
# Config
# -------------------
N_SETS = 100                    # traces per energy
TRACE_SAMPLES = 300_000         # must match long_trace_samples passed to lts.generate
DTYPE = np.float16
COMPRESSION_LEVEL = 15
ENERGIES = list(range(5, 35, 5))  # [5,10,15,20,25,30]
MAX_THREADS = 15

SIGNAL_THRESHOLD = 1.0  # pre-noise, pre-quant peak > threshold => signal


# -------------------
# I/O helpers: TRACES
# -------------------
def _shuffle_bytes(arr: np.ndarray) -> bytes:
    """Byte-shuffle for better compression (float16/float32)."""
    return arr.view(np.uint8).reshape(-1, arr.itemsize).T.tobytes()


def save_traces_to_zstd(traces, output_path: Path,
                        dtype=DTYPE, trace_shape=None, compression_level=COMPRESSION_LEVEL):
    """
    Save a list of arrays with identical shape=trace_shape and dtype to a .zst file.
    Layout: concatenated, byte-shuffled per trace.
    """
    if trace_shape is None:
        if not traces:
            raise ValueError("No traces to save and trace_shape not provided.")
        trace_shape = traces[0].shape

    all_data = bytearray()
    for trace in traces:
        if trace.shape != trace_shape:
            raise ValueError(f"Trace has wrong shape {trace.shape}, expected {trace_shape}")
        shuffled = _shuffle_bytes(np.asarray(trace, dtype=dtype))
        all_data.extend(shuffled)

    compressor = zstd.ZstdCompressor(level=compression_level)
    compressed_data = compressor.compress(bytes(all_data))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "wb") as f:
        f.write(compressed_data)


def load_traces_from_zstd(input_path: Path, n_traces: int,
                          dtype=DTYPE, trace_shape=None) -> np.ndarray:
    """
    Load stacked ndarray of shape (n_traces, *trace_shape) from a .zst file written by save_traces_to_zstd.
    """
    if trace_shape is None:
        raise ValueError("trace_shape must be provided to load traces.")

    def _unshuffle_bytes(data: bytes, dtype=dtype, shape=trace_shape) -> np.ndarray:
        itemsize = np.dtype(dtype).itemsize
        num_elements = int(np.prod(shape))
        reshaped = np.frombuffer(data, dtype=np.uint8).reshape(itemsize, num_elements).T
        unshuffled = reshaped.reshape(-1)
        return unshuffled.view(dtype).reshape(shape)

    decompressor = zstd.ZstdDecompressor()
    with open(input_path, "rb") as f:
        decompressed = decompressor.decompress(f.read())

    trace_size_bytes = int(np.prod(trace_shape)) * np.dtype(dtype).itemsize
    expected_size = n_traces * trace_size_bytes
    if len(decompressed) != expected_size:
        raise ValueError(f"Decompressed size {len(decompressed)} != expected {expected_size}")

    traces = []
    for i in range(n_traces):
        start = i * trace_size_bytes
        end = start + trace_size_bytes
        trace_bytes = decompressed[start:end]
        trace = _unshuffle_bytes(trace_bytes, dtype=dtype, shape=trace_shape)
        traces.append(trace)

    return np.stack(traces).astype(np.float32)


# -------------------
# I/O helpers: MASKS
# -------------------
def save_masks_to_zstd(masks: np.ndarray, output_path: Path,
                       mask_len: int, compression_level=COMPRESSION_LEVEL):
    """
    Save boolean masks of shape (N, mask_len) packed with np.packbits (little-endian), row by row.
    """
    if masks.dtype != np.bool_:
        masks = masks.astype(np.bool_)
    if masks.ndim != 2 or masks.shape[1] != mask_len:
        raise ValueError(f"Mask shape {masks.shape} does not match mask_len={mask_len}")

    bytes_per_row = (mask_len + 7) // 8
    rows = []
    for row in masks:
        packed = np.packbits(row, bitorder="little")
        if packed.size < bytes_per_row:
            pad = np.zeros(bytes_per_row - packed.size, dtype=np.uint8)
            packed = np.concatenate([packed, pad])
        rows.append(packed.tobytes())
    blob = b"".join(rows)

    comp = zstd.ZstdCompressor(level=compression_level)
    data = comp.compress(blob)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "wb") as f:
        f.write(data)


def load_masks_from_zstd(input_path: Path, n_masks: int, mask_len: int) -> np.ndarray:
    """
    Load boolean masks saved by save_masks_to_zstd.
    """
    bytes_per_row = (mask_len + 7) // 8
    expected = n_masks * bytes_per_row

    decomp = zstd.ZstdDecompressor()
    with open(input_path, "rb") as f:
        data = decomp.decompress(f.read())

    if len(data) != expected:
        raise ValueError(f"Decompressed mask size {len(data)} != expected {expected}")

    arr = np.frombuffer(data, dtype=np.uint8).reshape(n_masks, bytes_per_row)
    masks = np.unpackbits(arr, axis=1, bitorder="little", count=mask_len)
    return masks.astype(bool)

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


In [2]:
trace = load_traces_from_zstd(input_path = "/ceph/dwong/trigger_samples/v2_300k/traces_energy_0.zst", n_traces=100, trace_shape=(56, 300000))

In [3]:
trace.shape

(100, 56, 300000)

In [5]:
mask = load_masks_from_zstd(input_path = "/ceph/dwong/trigger_samples/v2_300k/masks_energy_10.zst", n_masks=100, mask_len=56)

In [5]:
mask[10]

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False,  True, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False])

In [6]:
from OptimumFilter import *

sampling_frequency = 3906250
vac_template = np.load("/home/dwong/DELight_mtr/trigger_study/archive/wk15/templates/vac_ch_template.npy")

noise_psd = np.load("/home/dwong/DELight_mtr/templates/noise_psd_from_MMC.npy")
vac_of = OptimumFilter(vac_template, noise_psd, sampling_frequency)

In [8]:
trace[10].shape

(56, 300000)

In [9]:
ampl, chisq = vac_of.sliding_fit(trace[10][1], hop=1)

In [10]:
print(np.max(ampl))

2.902744361784803


In [3]:
"""
Compute per-trace max OF amplitude with 56-way concurrency + tqdm progress bars.

- Uses OptimumFilter.sliding_fit(..., chisq_mode='none') for speed (we only need amplitude).
- Spawns a process pool with 56 workers (or fewer if fewer CPUs).
- Processes 56 traces of each repeat in parallel; iterates over 100 repeats.
- Shows a tqdm bar over repeats and an inner bar over traces.
- Returns and saves a (num_repeats, num_traces) array of maxima.

Assumptions
- `traces` is shaped (num_repeats, num_traces, num_samples), e.g. (100, 56, 300000), dtype float64/float32.
- Template and PSD are compatible with the window length used by OptimumFilter.

Usage example at bottom under __main__.
"""

from concurrent.futures import ProcessPoolExecutor
import numpy as np
import os
from OptimumFilter import OptimumFilter  # your v1.3 recommended
from tqdm.auto import tqdm

# Globals inside worker processes
_WORKER_OF = None
_REANCHOR_EVERY = None


def _worker_init(template: np.ndarray, noise_psd: np.ndarray, fs: float, reanchor_every: int | None):
    """Initializer runs once per worker. Builds a local OptimumFilter to avoid re-sending it."""
    global _WORKER_OF, _REANCHOR_EVERY
    _WORKER_OF = OptimumFilter(template, noise_psd, fs)
    _REANCHOR_EVERY = reanchor_every


def _max_amp_for_trace(trace_1d: np.ndarray) -> float:
    """Compute max amplitude for a single 1D trace using the worker's OptimumFilter."""
    x = np.ascontiguousarray(trace_1d, dtype=np.float64)
    amps, _ = _WORKER_OF.sliding_fit(x, hop=1, reanchor_every=_REANCHOR_EVERY, chisq_mode='none')
    return float(np.max(amps))


def compute_max_amplitudes(
    traces: np.ndarray,
    template: np.ndarray,
    noise_psd: np.ndarray,
    sampling_frequency: float,
    *,
    max_workers: int = 56,
    reanchor_every: int | None = None,
    out_path: str | None = None,
    progress: bool = True,
) -> np.ndarray:
    """
    Compute the max OF amplitude per trace with concurrency.

    Parameters
    ----------
    traces : array, shape (R, T, L)
        R repeats, T traces per repeat (e.g., 56), L samples per trace (e.g., 300000).
    template, noise_psd, sampling_frequency : passed to OptimumFilter
    max_workers : number of parallel worker processes (use <= #cores)
    reanchor_every : windows per block before re-anchoring (speed/accuracy trade-off). Use None to disable.
    out_path : if given, saves result as .npy
    progress : show tqdm progress bars

    Returns
    -------
    max_vals : array, shape (R, T)
    """
    traces = np.asarray(traces)
    if traces.ndim != 3:
        raise ValueError(f"traces must be 3D (repeats, traces, samples); got shape {traces.shape}")

    R, T, L = traces.shape

    # Ensure template & PSD are numpy arrays (copied to workers via initializer once)
    template = np.asarray(template, dtype=np.float64)
    noise_psd = np.asarray(noise_psd, dtype=np.float64)

    # Cap workers to CPU count
    n_workers = min(int(max_workers), os.cpu_count() or 1)

    max_vals = np.empty((R, T), dtype=np.float64)

    # Start pool once; reuse across repeats to amortize warm-up and compilation
    with ProcessPoolExecutor(
        max_workers=n_workers,
        initializer=_worker_init,
        initargs=(template, noise_psd, float(sampling_frequency), reanchor_every),
    ) as exe:
        outer_iter = range(R)
        if progress:
            outer_iter = tqdm(outer_iter, desc="Repeats", unit="repeat")

        for r in outer_iter:
            gen = (traces[r, t, :] for t in range(T))
            mapped = exe.map(_max_amp_for_trace, gen, chunksize=1)
            if progress:
                mapped = tqdm(mapped, total=T, desc=f"Repeat {r+1}/{R}", unit="trace", leave=False)
            row = [val for val in mapped]
            max_vals[r, :] = np.array(row, dtype=np.float64)

    if out_path:
        np.save(out_path, max_vals)
    return max_vals


if __name__ == "__main__":
    # --- Example wiring to your snippet ---
    sampling_frequency = 3_906_250
    vac_template = np.load("/home/dwong/DELight_mtr/trigger_study/archive/wk15/templates/vac_ch_template.npy")
    noise_psd = np.load("/home/dwong/DELight_mtr/templates/noise_psd_from_MMC.npy")

    # Expect `trace` in the environment with shape (100, 56, 300000)
    try:
        traces = trace  # noqa: F821 - provided by your session
    except NameError:
        raise RuntimeError("Variable 'trace' not found. Provide your data as a 3D array 'trace'.")

    max_vals = compute_max_amplitudes(
        traces=traces,
        template=vac_template,
        noise_psd=noise_psd,
        sampling_frequency=sampling_frequency,
        max_workers=56,
        reanchor_every=None,
        out_path="max_amplitudes_energy_0.npy",
        progress=True,
    )

    print("Saved max amplitudes to max_amplitudes.npy with shape:", max_vals.shape)


Repeats:   0%|          | 0/100 [00:00<?, ?repeat/s]

Repeat 1/100:   0%|          | 0/56 [00:00<?, ?trace/s]

Repeat 2/100:   0%|          | 0/56 [00:00<?, ?trace/s]

Repeat 3/100:   0%|          | 0/56 [00:00<?, ?trace/s]

Repeat 4/100:   0%|          | 0/56 [00:00<?, ?trace/s]

...

Repeat 97/100:   0%|          | 0/56 [00:00<?, ?trace/s]

Repeat 98/100:   0%|          | 0/56 [00:00<?, ?trace/s]

Repeat 99/100:   0%|          | 0/56 [00:00<?, ?trace/s]

Repeat 100/100:   0%|          | 0/56 [00:00<?, ?trace/s]

Saved max amplitudes to max_amplitudes.npy with shape: (100, 56)


In [5]:
def load_max_amplitudes(path="max_amplitudes.npy"):
    arr = np.load(path, mmap_mode="r")
    print("Loaded", path, "with shape", arr.shape)
    r_idx, t_idx = np.unravel_index(arr.argmax(), arr.shape)
    print("Global max:", float(arr[r_idx, t_idx]), "at (repeat, trace) =", (r_idx, t_idx))
    return arr

max_vals = load_max_amplitudes()


Loaded max_amplitudes.npy with shape (100, 56)
Global max: 6.385646056169698 at (repeat, trace) = (44, 37)
